## AWS Network Firewall로 Domain Filtering 검증

이 Notebook은 `agentcore-browser-firewall.yaml`로 배포한 AWS Network Firewall이 AgentCore Browser 세션의 domain을 올바르게 filtering하는지 검증합니다.

Firewall은 다음 세 가지 범주를 적용합니다.
- **Allowlist** - 명시적으로 허용된 domain(예: `example.com`, `github.com`)
- **Denylist** - 명시적으로 차단된 domain(예: `facebook.com`, `twitter.com`)
- **Default deny** - 두 목록에 모두 없는 domain 차단

### 사전 요구 사항

1. `agentcore-browser-firewall.yaml` CloudFormation stack 배포
2. Dependency를 설치하고 **kernel 다시 시작**:

In [ ]:
!pip install -qU -r requirements.txt

### 1. 설정

CloudFormation stack output에서 Browser ID를 가져와 client를 초기화합니다.

In [ ]:
import boto3
from urllib.parse import urlparse
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

session = boto3.Session()
REGION = session.region_name
browser_client = boto3.client("bedrock-agentcore")

# CloudFormation output에서 BROWSER_ID 가져오기
cfn = boto3.client("cloudformation")
outputs = cfn.describe_stacks(StackName="agentcore-browser-firewall")["Stacks"][0]["Outputs"]
BROWSER_ID = next(o["OutputValue"] for o in outputs if o["OutputKey"] == "BrowserToolCustomOutput")
print(f"Browser ID: {BROWSER_ID}")

### 2. 브라우저 세션 시작

세션을 시작하고 Playwright 연결에 사용할 SigV4-signed WebSocket URL을 생성합니다.

In [ ]:
response = browser_client.start_browser_session(browserIdentifier=BROWSER_ID)
session_id = response["sessionId"]
ws_url = f"wss://bedrock-agentcore.{REGION}.amazonaws.com/browser-streams/{BROWSER_ID}/sessions/{session_id}/automation"
print(f"Session ID: {session_id}")

# SigV4로 WebSocket URL 서명
credentials = session.get_credentials()
https_url = ws_url.replace("wss://", "https://")
parsed = urlparse(https_url)
request = AWSRequest(method="GET", url=https_url, headers={"host": parsed.netloc})
SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(request)
headers = {k: v for k, v in request.headers.items()}

### 3. Domain filtering 검증 실행

Playwright로 연결하고 각 범주의 domain 탐색을 시도합니다.

아래 테스트 URL은 CloudFormation template의 기본 `AllowedDomains` 및 `DeniedDomains` parameter와 일치합니다. 해당 parameter를 사용자 지정했다면 URL도 그에 맞게 수정하세요.

In [ ]:
from playwright.async_api import async_playwright

# (url, 범주, 허용 여부)
tests = [
    ("https://example.com", "ALLOWLIST", True),
    ("https://github.com", "ALLOWLIST", True),
    ("https://wikipedia.org", "ALLOWLIST", True),
    ("https://facebook.com", "DENYLIST", False),
    ("https://twitter.com", "DENYLIST", False),
    ("https://randomsite12345.com", "UNLISTED", False),
]

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    print("=" * 60)
    print("DOMAIN FILTERING VERIFICATION RESULTS")
    print("=" * 60)

    results = []
    for url, category, should_allow in tests:
        try:
            resp = await page.goto(url, timeout=10000, wait_until="domcontentloaded")
            allowed = resp is not None and resp.status < 400
            passed = allowed == should_allow
            status_str = f"HTTP {resp.status}" if resp else "No response"
            result = "PASS" if passed else "FAIL"
            label = "Allowed" if allowed else "Blocked"
            print(f"{result}: {url} ({category}) - {label} [{status_str}]")
            results.append(passed)
        except Exception as e:
            passed = not should_allow
            result = "PASS" if passed else "FAIL"
            print(f"{result}: {url} ({category}) - Blocked ({type(e).__name__})")
            results.append(passed)

    print("=" * 60)
    print(f"Results: {sum(results)}/{len(results)} tests passed")
    await browser.close()

### 4. 세션 중지

In [ ]:
browser_client.stop_browser_session(browserIdentifier=BROWSER_ID, sessionId=session_id)
print(f"Session {session_id} stopped")

### 문제 해결

테스트가 예상대로 동작하지 않으면 CloudWatch에서 Network Firewall log를 확인하세요.

```
/aws/network-firewall/agentcore-browser-firewall/alert
/aws/network-firewall/agentcore-browser-firewall/flow
```

Alert log에는 일치한 rule이 표시됩니다. Flow log에는 firewall을 통과하는 모든 traffic이 표시되므로 예기치 않게 차단되거나 허용된 domain을 디버깅하는 데 유용합니다.